In [ ]:
import os
import random
import numpy as np
import torch
import optuna
import pandas as pd

from sb3_contrib import RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback

from agent import FloodWarningEnv

from optuna.visualization import plot_slice, plot_param_importances

# --- Curriculum settings ---
SEVERE_PROB_START = 0.9   # start with 90% severe episodes
SEVERE_PROB_END   = 0.0   # decay to 0% by end of training
CURRICULUM_END_FRAC = 0.5 # point which curriculum finishes through training

STUDY_NAME = "flood_warning_tuning_lstm"
OUTPUT_DIR = "./tuning/tuning_lstm/"
DB_PATH = "sqlite:///./tuning/tuning_lstm/optuna_study_lstm.db"

class CurriculumCallback(BaseCallback):
    def __init__(self, envs, total_steps, start_prob, end_prob, end_frac, verbose=0):
        super().__init__(verbose)
        self.envs = envs
        self.total_steps = total_steps
        self.start_prob = start_prob
        self.end_prob = end_prob
        self.curriculum_steps = int(total_steps * end_frac)

    def _on_step(self) -> bool:
        progress = min(self.num_timesteps / self.curriculum_steps, 1.0)
        severe_prob = self.start_prob + progress * (self.end_prob - self.start_prob)

        for env in self.envs:
            env.set_severe_prob(severe_prob)

        return True


def make_env(severe_prob):
    return lambda: FloodWarningEnv(severe_prob=severe_prob)

### Hyperparameter Tuning

In [3]:
SEED = 1
N_TRIALS_PER_RUN = 17
TRAIN_STEPS = 300_000
N_EVAL_EPISODES = 50

os.makedirs(OUTPUT_DIR, exist_ok=True)

def objective(trial):
    # --- Hyperparameters ---
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    n_steps = trial.suggest_categorical("n_steps", [256, 512, 1024])
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
    n_epochs = trial.suggest_int("n_epochs", 5, 20)
    ent_coef = trial.suggest_float("ent_coef", 1e-4, 0.1, log=True)

    # --- Seeds ---
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    # --- Environments ---
    env = make_vec_env(
        make_env(severe_prob=SEVERE_PROB_START),
        n_envs=8,
        seed=SEED
    )

    eval_env = make_vec_env(
        make_env(severe_prob=0.0),  # no curriculum for evaluation
        n_envs=1,
        seed=SEED
    )

    # Get underlying envs for curriculum updates
    train_envs = [env.envs[i].env for i in range(8)]

    # --- Model ---
    model = RecurrentPPO(
        "MlpLstmPolicy",
        env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        batch_size=batch_size,
        n_epochs=n_epochs,
        ent_coef=ent_coef,
        gamma=0.99,
        verbose=0,
        seed=SEED,
    )

    # --- Callbacks ---
    curriculum_callback = CurriculumCallback(
        envs=train_envs,
        total_steps=TRAIN_STEPS,
        start_prob=SEVERE_PROB_START,
        end_prob=SEVERE_PROB_END,
        end_frac=CURRICULUM_END_FRAC,
        verbose=0,
    )

    eval_callback = EvalCallback(
        eval_env,
        eval_freq=25_000,
        n_eval_episodes=N_EVAL_EPISODES,
        deterministic=True,
        verbose=0,
    )

    # --- Training ---
    model.learn(
        total_timesteps=TRAIN_STEPS,
        callback=[curriculum_callback, eval_callback]
    )

    mean_reward = eval_callback.last_mean_reward

    env.close()
    eval_env.close()

    return mean_reward


study = optuna.create_study(
    direction="maximize",
    storage=DB_PATH,
    study_name=STUDY_NAME,
    load_if_exists=True
)

study.optimize(objective, n_trials=N_TRIALS_PER_RUN)

print("Best trial:")
print(f"  Mean reward: {study.best_trial.value:.4f}")
print(f"  Params: {study.best_trial.params}")

study.trials_dataframe().to_csv(
    os.path.join(OUTPUT_DIR, "study_results.csv"),
    index=False
)

print("Study results saved to " + os.path.join(OUTPUT_DIR, "study_results.csv"))

[I 2026-04-19 19:46:32,903] Using an existing study with name 'flood_warning_tuning_lstm' instead of creating a new one.
[I 2026-04-19 20:22:38,792] Trial 65 finished with value: -327.56 and parameters: {'learning_rate': 0.0005544904129060824, 'n_steps': 256, 'batch_size': 64, 'n_epochs': 15, 'ent_coef': 0.0015550915385614341}. Best is trial 22 with value: -34.84.
[I 2026-04-19 20:44:37,070] Trial 66 finished with value: -280.25 and parameters: {'learning_rate': 0.0008522115418651058, 'n_steps': 512, 'batch_size': 64, 'n_epochs': 9, 'ent_coef': 0.0003018091825208737}. Best is trial 22 with value: -34.84.
[I 2026-04-19 21:23:03,816] Trial 67 finished with value: -186.64 and parameters: {'learning_rate': 0.0006843635828383312, 'n_steps': 512, 'batch_size': 64, 'n_epochs': 17, 'ent_coef': 0.041482093266188326}. Best is trial 22 with value: -34.84.
[I 2026-04-19 22:25:13,747] Trial 68 finished with value: -100.98 and parameters: {'learning_rate': 0.0005194538201537549, 'n_steps': 1024, 'ba

Best trial:
  Mean reward: -34.8400
  Params: {'learning_rate': 0.0004636634545890192, 'n_steps': 512, 'batch_size': 128, 'n_epochs': 16, 'ent_coef': 0.04367100498590175}
Study results saved to ./tuning/tuning_lstm/study_results.csv


In [4]:
# Check completed runs
study = optuna.load_study(study_name=STUDY_NAME, storage=DB_PATH)
n_completed = len(study.get_trials(states=[optuna.trial.TrialState.COMPLETE]))
print(n_completed)
n_failed = len(study.get_trials(states=[optuna.trial.TrialState.FAIL]))
print(n_failed)

75
7


### Hyperparameter sensitivity analysis

In [ ]:
study = optuna.load_study(study_name=STUDY_NAME, storage=DB_PATH)

# Slice plot (parameter vs objective)
fig_slice = plot_slice(study)
fig_slice.write_image(os.path.join(OUTPUT_DIR, "slice_plot.png"))
fig_slice.show()

# Parameter importance plot
fig_importance = plot_param_importances(study)
fig_importance.write_image(os.path.join(OUTPUT_DIR, "param_importances.png"))
fig_importance.show() 

### Full training using best combination

In [5]:
study = optuna.load_study(study_name=STUDY_NAME, storage=DB_PATH)

best_trial = study.best_trial

print("Best reward:", best_trial.value)
print("Best params:", best_trial.params)

LEARNING_RATE = best_trial.params["learning_rate"]
N_STEPS = best_trial.params["n_steps"]
BATCH_SIZE = best_trial.params["batch_size"]
N_EPOCHS = best_trial.params["n_epochs"]
ENT_COEF = best_trial.params["ent_coef"]

Best reward: -34.84
Best params: {'learning_rate': 0.0004636634545890192, 'n_steps': 512, 'batch_size': 128, 'n_epochs': 16, 'ent_coef': 0.04367100498590175}


In [ ]:
SEED = 1 # 1, 2, 3
TRAIN_STEPS = 5_000_000
N_EVAL_EPISODES = 200

OUTPUT_DIR = "./models/lstm/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Training envs start with severe_prob=SEVERE_PROB_START
env = make_vec_env(make_env(severe_prob=SEVERE_PROB_START), n_envs=8, seed=SEED)

# Eval env always uses natural distribution (no curriculum)
eval_env = make_vec_env(make_env(severe_prob=0.0), n_envs=1, seed=SEED)

train_envs = [env.envs[i].env for i in range(8)]

# Agent
model = RecurrentPPO(
    "MlpLstmPolicy",
    env,
    learning_rate=LEARNING_RATE,
    n_steps=N_STEPS,
    batch_size=BATCH_SIZE,
    n_epochs=N_EPOCHS,
    ent_coef=ENT_COEF,
    gamma=0.99,
    verbose=1,
    seed=SEED,
)

# Callbacks
curriculum_callback = CurriculumCallback(
    envs=train_envs,
    total_steps=TRAIN_STEPS,
    start_prob=SEVERE_PROB_START,
    end_prob=SEVERE_PROB_END,
    end_frac=CURRICULUM_END_FRAC,
    verbose=1,
)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=OUTPUT_DIR,
    log_path=OUTPUT_DIR,
    eval_freq=25_000,
    n_eval_episodes=N_EVAL_EPISODES,
    deterministic=True,
    verbose=1,
)

model.learn(
    total_timesteps=TRAIN_STEPS,
    callback=[curriculum_callback, eval_callback]
)

model.save(os.path.join(OUTPUT_DIR, "final_model"))

print(f"Training complete.")
print(f"Best mean reward: {eval_callback.best_mean_reward:.4f}")
print(f"Final mean reward: {eval_callback.last_mean_reward:.4f}")
print(f"Model saved to {OUTPUT_DIR}")

env.close()
eval_env.close()

Using cpu device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 200      |
|    ep_rew_mean     | -870     |
| time/              |          |
|    fps             | 451      |
|    iterations      | 1        |
|    time_elapsed    | 9        |
|    total_timesteps | 4096     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 200         |
|    ep_rew_mean          | -930        |
| time/                   |             |
|    fps                  | 78          |
|    iterations           | 2           |
|    time_elapsed         | 104         |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.018308515 |
|    clip_fraction        | 0.245       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.37       |
|    explained_variance   | -0.00366    |
|    learning